# ⚡ Revio Pulse AI: Autonomous CX & Reputation Intelligence Engine
### AI/ML Engineer Intern Assessment Deliverable | Cynbit Technologies (`cynbit.in`)

**Author:** AI/ML Engineer Intern Candidate  
**Target Product:** Cynbit Revio SaaS Suite  
**Focus:** Aspect-Based Sentiment Analysis (ABSA), Churn Risk Forecasting, and Autonomous Resolution Dispatch

[Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)

## 1. Executive Summary & Objective
Cynbit Technologies builds modern SaaS applications including **Revio**, their AI-first review and reputation intelligence platform.  
While existing reputation solutions offer basic 3-class sentiment (positive/neutral/negative) and static templates, **Revio Pulse AI** introduces:
1. **Multi-Aspect Sentiment Analysis (ABSA):** Pinpointing specific business facets (Billing, Reliability, UI/UX, Support).
2. **Churn Risk & Urgency Forecasting:** Quantifying defection probability and enforcing SLA routing (< 1 hour for P0 risks).
3. **Context-Aware Response Synthesis:** Generating brand-safe, empathetic resolution drafts that validate customer concerns.
4. **Automated Ticket Dispatch:** Instantly piping critical reviews into Jira/Linear and Slack for immediate cross-functional resolution.

In [ ]:
# Setup & Core Imports
import re
import json
from collections import Counter
from typing import Dict, List, Any

print("Environment initialized successfully for Revio Pulse AI!")

## 2. Aspect-Based Sentiment Analysis (ABSA) Implementation

In [ ]:
ASPECT_TAXONOMY = {
    "Billing & Pricing": ["billing", "price", "pricing", "charged", "fee", "cost", "refund", "invoice", "chargeback"],
    "Platform Reliability & Uptime": ["crash", "crashed", "downtime", "down", "bug", "error", "locked out", "latency", "slow", "broken"],
    "Customer Support & Responsiveness": ["support", "service", "helpdesk", "ticket", "response", "unresponsive", "queue"],
    "UI/UX & Product Usability": ["ui", "ux", "interface", "design", "dashboard", "sleek", "intuitive", "clunky"],
    "Features & Integrations": ["feature", "integration", "webhook", "api", "slack", "csv", "sync", "export"]
}

POSITIVE_WORDS = {"amazing", "great", "excellent", "outstanding", "accurate", "saves", "clean", "sleek", "recommend", "love", "solid"}
NEGATIVE_WORDS = {"horrible", "crashed", "crash", "locked out", "unresponsive", "lost", "avoid", "deceptive", "hidden", "broke", "broken", "slow"}

def analyze_absa(text: str, rating: float):
    text_lower = text.lower()
    sentences = re.split(r"[.!?]+", text_lower)
    results = {}
    detected = []
    
    for aspect, kws in ASPECT_TAXONOMY.items():
        matched = [k for k in kws if k in text_lower]
        if not matched:
            continue
        detected.append(aspect)
        score = 0.0
        count = 0
        for s in sentences:
            if any(k in s for k in matched):
                words = re.findall(r"\b[\w'-]+\b", s)
                for w in words:
                    if w in POSITIVE_WORDS:
                        score += 1.0
                        count += 1
                    elif w in NEGATIVE_WORDS:
                        score -= 1.0
                        count += 1
        norm_score = max(min(score / count, 1.0), -1.0) if count > 0 else (rating - 3.0) / 2.0
        results[aspect] = {
            "score": round(norm_score, 2),
            "polarity": "Positive" if norm_score >= 0.2 else "Negative" if norm_score <= -0.2 else "Neutral",
            "matched": matched
        }
    return {"detected_aspects": detected, "aspect_breakdown": results}

## 3. Churn Risk & Urgency Scoring Engine

In [ ]:
def calculate_churn_urgency(text: str, rating: float, verified: bool = True):
    text_lower = text.lower()
    score = 45 if rating == 1 else 28 if rating == 2 else 10 if rating == 3 else 2
    dept = "Customer Success"
    signals = []
    
    if any(w in text_lower for w in ["legal", "lawyer", "chargeback", "fraudulent"]):
        score += 35
        dept = "Legal & Risk Compliance"
        signals.append("Legal/Chargeback Threat")
    if any(w in text_lower for w in ["cancel", "cancelling", "switch to", "moving to", "birdeye", "competitor"]):
        score += 25
        dept = "Executive Escalations / Retention"
        signals.append("Competitor Defection Risk")
    if any(w in text_lower for w in ["lost", "transactions", "stole", "hidden fee", "refund"]):
        score += 20
        signals.append("Financial Impact")
    if any(w in text_lower for w in ["crash", "crashed", "outage", "locked out", "peak hours"]):
        score += 22
        signals.append("Production Outage")
        
    churn_score = min(score, 100)
    urgency = "CRITICAL" if churn_score >= 75 else "HIGH" if churn_score >= 50 else "MEDIUM" if churn_score >= 25 else "LOW"
    sla = "< 1 hour" if urgency == "CRITICAL" else "< 4 hours" if urgency == "HIGH" else "< 24 hours"
    
    return {
        "churn_risk_score": churn_score,
        "urgency_level": urgency,
        "sla_target": sla,
        "escalation_department": dept,
        "requires_escalation": urgency in ["CRITICAL", "HIGH"],
        "signals": signals
    }

## 4. Context-Aware Autonomous Response Synthesis

In [ ]:
def synthesize_response(reviewer_name: str, platform: str, rating: float, text: str, absa: dict, urgency: dict):
    first_name = reviewer_name.split()[0]
    aspects = ", ".join(absa["detected_aspects"]) if absa["detected_aspects"] else "the service"
    dept = urgency["escalation_department"]
    
    if rating >= 4:
        return (
            f"Hi {first_name},\n\nThank you for sharing your review on {platform}! "
            f"We are thrilled to hear that Revio is adding real value to your workflow. "
            f"Our engineering team in Jaipur is committed to continuous innovation.\n\n"
            f"Warm regards,\nThe Revio & Cynbit Team"
        )
    else:
        return (
            f"Hi {first_name},\n\nWe sincerely apologize for the frustration caused regarding {aspects}. "
            f"This falls short of our standards at Cynbit. We have immediately escalated your ticket "
            f"to our {dept} (Priority #{abs(hash(text)) % 9000 + 1000}) with SLA target {urgency['sla_target']}.\n\n"
            f"Please connect with our executive team directly at escalations@cynbit.in so we can make this right.\n\n"
            f"Respectfully,\nLeadership Team | Revio by Cynbit"
        )

## 5. End-to-End Pipeline Evaluation on Live Benchmark Cases

In [ ]:
sample_case = {
    "name": "Aarav Sharma",
    "platform": "Google Maps",
    "rating": 1.0,
    "text": "Horrible experience. The system crashed during peak billing hours and our staff was locked out for 4 hours. No phone support available, only an unresponsive ticket queue. We lost over $4,000 in customer transactions. Avoid this platform!"
}

absa = analyze_absa(sample_case["text"], sample_case["rating"])
urgency = calculate_churn_urgency(sample_case["text"], sample_case["rating"])
draft = synthesize_response(sample_case["name"], sample_case["platform"], sample_case["rating"], sample_case["text"], absa, urgency)

print("=" * 70)
print(f"ANALYSIS FOR: {sample_case['name']} ({sample_case['platform']})")
print("=" * 70)
print(f"Detected Aspects: {absa['detected_aspects']}")
print(f"Churn Risk: {urgency['churn_risk_score']}% | Urgency: [{urgency['urgency_level']}] | SLA: {urgency['sla_target']}")
print(f"Escalation Department: {urgency['escalation_department']}")
print("\n--- AI SYNTHESIZED RESPONSE DRAFT ---")
print(draft)